# PyBullet Ping-Pong Ball Bounce: 12-View Physically Grounded Capture

This notebook extends `pybullet_tabletop_multiview.ipynb` from four views to a synchronized 12-view camera rig. The underlying simulation is unchanged in spirit: PyBullet runs in SI units and integrates gravity, rigid-body contact, restitution, Coulomb-style friction, rolling/spinning friction, and quadratic aerodynamic drag for a regulation ping-pong ball bouncing on a tabletop.

The main change is observational: instead of front/side/back/top only, we sample eight azimuth views around the table, two overhead views, and two low oblique views. That makes the dataset closer to a multi-view geometry setup: every frame is rendered from a known virtual camera at the same physical time, so the views are different projections of one shared 3D trajectory. The low cameras are useful for seeing contact and rebound near the tabletop without using a physically occluded bottom view. The scene also includes a simple room shell with floor, walls, ceiling, and trim landmarks so reconstruction has stable visual context around the moving ball.


In [1]:
import os
import sys
from pathlib import Path

# This notebook is intended to run with the repository-local virtual environment.
assert "phys_sim" in sys.executable, (
    f"Expected the phys_sim virtual environment, but got: {sys.executable}\n"
    "In Jupyter, select the kernel named 'phys_sim'."
)

import numpy as np
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
from IPython.display import Video, display

print(f"Python executable: {sys.executable}")
print(f"PyBullet data path: {pybullet_data.getDataPath()}")

Python executable: /Users/dkoffical/Documents/GitHub/cs231n_project_phys4d/phys_sim/bin/python
PyBullet data path: /Users/dkoffical/Documents/GitHub/cs231n_project_phys4d/phys_sim/lib/python3.10/site-packages/pybullet_data


pybullet build time: May  3 2026 11:26:45


## Scenario Parameters

All distances are meters, mass is kilograms, and time is seconds. The ball uses regulation ping-pong dimensions: 40 mm diameter and 2.7 g mass. The default bounce starts vertically with a small lateral drift, which makes the 12-view rig useful because the same 3D motion is visible from multiple angles.


In [2]:
OUTPUT_DIR = Path("pybullet_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Simulation fidelity
GRAVITY = -9.80665
SIM_HZ = 480
TIME_STEP = 1.0 / SIM_HZ
VIDEO_FPS = 60
STEPS_PER_FRAME = SIM_HZ // VIDEO_FPS
DURATION_SEC = 4.0
N_STEPS = int(DURATION_SEC * SIM_HZ)

# Room geometry: the 12 cameras sit inside this simple visual shell.
ROOM_HALF_X = 2.20
ROOM_HALF_Y = 2.20
ROOM_HEIGHT = 2.85
ROOM_WALL_THICKNESS = 0.04
ROOM_FLOOR_THICKNESS = 0.04

# Table geometry: tabletop center z plus half thickness gives the upper surface.
TABLE_LENGTH = 1.20
TABLE_WIDTH = 0.80
TABLE_THICKNESS = 0.06
TABLE_TOP_Z = 0.75
TABLE_CENTER_Z = TABLE_TOP_Z - TABLE_THICKNESS / 2.0

# Regulation ping-pong ball geometry and material.
BALL_RADIUS = 0.020
BALL_MASS = 0.0027
BALL_START_Z = TABLE_TOP_Z + 0.80
BALL_START_XY = [-0.12, -0.04]
BALL_INITIAL_LINEAR_VELOCITY = [0.08, 0.03, 0.0]
BALL_INITIAL_ANGULAR_VELOCITY = [0.0, 0.0, 0.0]

# A regulation ball dropped from 305 mm should rebound about 240-260 mm on a standard block,
# which implies an effective coefficient of restitution near sqrt(0.25 / 0.305) ~= 0.91.
BALL_RESTITUTION = 0.93
BALL_LATERAL_FRICTION = 0.20
BALL_ROLLING_FRICTION = 0.0005
BALL_SPINNING_FRICTION = 0.0005

TABLE_RESTITUTION = 0.93
TABLE_LATERAL_FRICTION = 0.35

# Aerodynamics matter for a very light 40 mm ball.
AIR_DENSITY = 1.225
DRAG_COEFFICIENT = 0.47
BALL_CROSS_SECTION = np.pi * BALL_RADIUS ** 2

# Rendering
IMG_WIDTH = 960
IMG_HEIGHT = 544

print(f"Writing videos and trajectory files to: {OUTPUT_DIR.resolve()}")

Writing videos and trajectory files to: /Users/dkoffical/Documents/GitHub/cs231n_project_phys4d/pybullet_outputs


## Physical Law Grounding

The simulation is built around a small set of classical mechanics assumptions that are easy to defend in a writeup:

- **Newtonian dynamics:** PyBullet advances the rigid body state using Newton's second law, `F = m a`, with a fixed time step. The ball's position and velocity evolve from the net force applied at each step.
- **Uniform gravity:** The vertical force is `m g`, with `g = 9.80665 m/s^2` downward. This gives the parabolic free-flight arcs between bounces.
- **Rigid-body contact:** The table and ball are treated as rigid bodies. During impact, PyBullet solves contact constraints so the ball does not pass through the tabletop.
- **Coefficient of restitution:** Restitution controls how much normal velocity is preserved after impact. A ping-pong ball has a high rebound, so the ball and table use restitution near `0.93`; rebound height is approximately proportional to `e^2` in an ideal vertical bounce.
- **Frictional contact:** Lateral, rolling, and spinning friction dissipate tangential motion and angular motion at the table contact. This is why the lateral drift and spin do not remain perfectly constant.
- **Quadratic air drag:** A light ping-pong ball is strongly affected by air resistance, so the notebook applies `F_d = -0.5 rho C_d A |v| v`. This force opposes motion and grows with speed squared.
- **Multi-view projection:** The 12 cameras do not change the physics. They measure the same simulated state through different known view/projection matrices, which is the same principle used in calibrated multi-camera reconstruction.

A concise writeup framing: *we simulate one physically consistent 3D trajectory, then render it through twelve calibrated virtual cameras. The camera diversity improves visual coverage while the underlying motion remains constrained by gravity, contact impulses, restitution, friction, and aerodynamic drag.*


In [3]:
def connect_pybullet():
    """Create a clean DIRECT PyBullet connection for headless notebook rendering."""
    if p.isConnected():
        p.disconnect()
    client = p.connect(p.DIRECT)
    p.resetSimulation(physicsClientId=client)
    p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=client)
    p.setGravity(0, 0, GRAVITY, physicsClientId=client)
    p.setTimeStep(TIME_STEP, physicsClientId=client)
    p.setPhysicsEngineParameter(
        fixedTimeStep=TIME_STEP,
        numSolverIterations=150,
        numSubSteps=2,
        deterministicOverlappingPairs=1,
        physicsClientId=client,
    )
    return client


def add_visual_box(client, half_extents, position, rgba_color):
    """Add static visual room geometry without introducing extra contacts."""
    visual = p.createVisualShape(
        p.GEOM_BOX,
        halfExtents=half_extents,
        rgbaColor=rgba_color,
        physicsClientId=client,
    )
    return p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=-1,
        baseVisualShapeIndex=visual,
        basePosition=position,
        physicsClientId=client,
    )


def create_room_shell(client):
    """Create a surrounding room with floor, walls, ceiling, and visual anchors."""
    floor_color = [0.62, 0.66, 0.63, 1.0]
    ceiling_color = [0.82, 0.84, 0.80, 1.0]
    front_back_color = [0.70, 0.76, 0.80, 1.0]
    side_color = [0.78, 0.73, 0.66, 1.0]
    trim_color = [0.35, 0.38, 0.36, 1.0]
    accent_blue = [0.32, 0.45, 0.70, 1.0]
    accent_green = [0.36, 0.55, 0.42, 1.0]

    floor_z = -ROOM_FLOOR_THICKNESS / 2.0
    ceiling_z = ROOM_HEIGHT + ROOM_WALL_THICKNESS / 2.0
    wall_z = ROOM_HEIGHT / 2.0

    room_ids = [
        add_visual_box(
            client,
            [ROOM_HALF_X, ROOM_HALF_Y, ROOM_FLOOR_THICKNESS / 2.0],
            [0.0, 0.0, floor_z],
            floor_color,
        ),
        add_visual_box(
            client,
            [ROOM_HALF_X, ROOM_HALF_Y, ROOM_WALL_THICKNESS / 2.0],
            [0.0, 0.0, ceiling_z],
            ceiling_color,
        ),
        add_visual_box(
            client,
            [ROOM_HALF_X, ROOM_WALL_THICKNESS / 2.0, ROOM_HEIGHT / 2.0],
            [0.0, -ROOM_HALF_Y, wall_z],
            front_back_color,
        ),
        add_visual_box(
            client,
            [ROOM_HALF_X, ROOM_WALL_THICKNESS / 2.0, ROOM_HEIGHT / 2.0],
            [0.0, ROOM_HALF_Y, wall_z],
            front_back_color,
        ),
        add_visual_box(
            client,
            [ROOM_WALL_THICKNESS / 2.0, ROOM_HALF_Y, ROOM_HEIGHT / 2.0],
            [-ROOM_HALF_X, 0.0, wall_z],
            side_color,
        ),
        add_visual_box(
            client,
            [ROOM_WALL_THICKNESS / 2.0, ROOM_HALF_Y, ROOM_HEIGHT / 2.0],
            [ROOM_HALF_X, 0.0, wall_z],
            side_color,
        ),
    ]

    # Thin trim and two color panels give the multiview learner fixed room cues.
    room_ids.extend([
        add_visual_box(client, [ROOM_HALF_X, 0.018, 0.025], [0.0, -ROOM_HALF_Y + 0.025, 0.16], trim_color),
        add_visual_box(client, [ROOM_HALF_X, 0.018, 0.025], [0.0, ROOM_HALF_Y - 0.025, 0.16], trim_color),
        add_visual_box(client, [0.018, ROOM_HALF_Y, 0.025], [-ROOM_HALF_X + 0.025, 0.0, 0.16], trim_color),
        add_visual_box(client, [0.018, ROOM_HALF_Y, 0.025], [ROOM_HALF_X - 0.025, 0.0, 0.16], trim_color),
        add_visual_box(client, [0.30, 0.012, 0.22], [-0.65, ROOM_HALF_Y - 0.045, 1.25], accent_blue),
        add_visual_box(client, [0.012, 0.26, 0.18], [ROOM_HALF_X - 0.045, -0.55, 1.10], accent_green),
    ])
    return room_ids


def create_tabletop_world(client):
    """Build a rigid tabletop, room shell, and spherical ball in SI units."""
    floor_collision = p.createCollisionShape(p.GEOM_PLANE, physicsClientId=client)
    p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=floor_collision,
        baseVisualShapeIndex=-1,
        physicsClientId=client,
    )
    room_ids = create_room_shell(client)

    table_collision = p.createCollisionShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        physicsClientId=client,
    )
    table_visual = p.createVisualShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        rgbaColor=[0.58, 0.40, 0.24, 1.0],
        physicsClientId=client,
    )
    table_id = p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=table_collision,
        baseVisualShapeIndex=table_visual,
        basePosition=[0, 0, TABLE_CENTER_Z],
        physicsClientId=client,
    )
    p.changeDynamics(
        table_id,
        -1,
        restitution=TABLE_RESTITUTION,
        lateralFriction=TABLE_LATERAL_FRICTION,
        physicsClientId=client,
    )

    ball_collision = p.createCollisionShape(
        p.GEOM_SPHERE,
        radius=BALL_RADIUS,
        physicsClientId=client,
    )
    ball_visual = p.createVisualShape(
        p.GEOM_SPHERE,
        radius=BALL_RADIUS,
        rgbaColor=[0.08, 0.38, 0.90, 1.0],
        physicsClientId=client,
    )
    ball_id = p.createMultiBody(
        baseMass=BALL_MASS,
        baseCollisionShapeIndex=ball_collision,
        baseVisualShapeIndex=ball_visual,
        basePosition=[BALL_START_XY[0], BALL_START_XY[1], BALL_START_Z],
        physicsClientId=client,
    )
    p.changeDynamics(
        ball_id,
        -1,
        restitution=BALL_RESTITUTION,
        lateralFriction=BALL_LATERAL_FRICTION,
        rollingFriction=BALL_ROLLING_FRICTION,
        spinningFriction=BALL_SPINNING_FRICTION,
        physicsClientId=client,
    )
    p.resetBaseVelocity(
        ball_id,
        linearVelocity=BALL_INITIAL_LINEAR_VELOCITY,
        angularVelocity=BALL_INITIAL_ANGULAR_VELOCITY,
        physicsClientId=client,
    )

    # Give the renderer a directional light that makes the table/ball readable.
    p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0, physicsClientId=client)
    return table_id, ball_id, room_ids


def apply_ping_pong_air_drag(client, ball_id):
    """Apply quadratic drag: Fd = -0.5 * rho * Cd * A * |v| * v."""
    lin_vel, _ = p.getBaseVelocity(ball_id, physicsClientId=client)
    velocity = np.asarray(lin_vel, dtype=float)
    speed = np.linalg.norm(velocity)
    if speed == 0:
        return
    drag = -0.5 * AIR_DENSITY * DRAG_COEFFICIENT * BALL_CROSS_SECTION * speed * velocity
    pos, _ = p.getBasePositionAndOrientation(ball_id, physicsClientId=client)
    p.applyExternalForce(
        ball_id,
        -1,
        drag.tolist(),
        pos,
        flags=p.WORLD_FRAME,
        physicsClientId=client,
    )


def orbit_camera_eye(azimuth_deg, radius=1.75, height=1.15):
    """Return an eye point on a circle around the tabletop target."""
    theta = np.deg2rad(azimuth_deg)
    return [radius * np.sin(theta), -radius * np.cos(theta), height]


def camera_specs():
    """Eight evenly spaced orbit views, two overhead views, and two low oblique views."""
    orbit_views = [
        ("front", 0),
        ("front_right", 45),
        ("right", 90),
        ("back_right", 135),
        ("back", 180),
        ("back_left", 225),
        ("left", 270),
        ("front_left", 315),
    ]
    specs = {
        name: {"eye": orbit_camera_eye(angle), "up": [0, 0, 1], "fov": 48}
        for name, angle in orbit_views
    }
    specs.update({
        "top": {"eye": [0.0, 0.0, 2.65], "up": [0, 1, 0], "fov": 42},
        "top_oblique": {"eye": [0.75, -0.75, 2.35], "up": [0, 0, 1], "fov": 46},
        "low_front_left": {"eye": [-0.90, -1.35, TABLE_TOP_Z + 0.12], "up": [0, 0, 1], "fov": 40},
        "low_back_right": {"eye": [0.90, 1.35, TABLE_TOP_Z + 0.12], "up": [0, 0, 1], "fov": 40},
    })
    return specs


def camera_matrices(view_name):
    target = [0, 0, TABLE_TOP_Z + 0.20]
    spec = CAMERA_SPECS[view_name]
    view = p.computeViewMatrix(spec["eye"], target, spec["up"])
    proj = p.computeProjectionMatrixFOV(
        fov=spec["fov"],
        aspect=IMG_WIDTH / IMG_HEIGHT,
        nearVal=0.02,
        farVal=5.0,
    )
    return view, proj


CAMERA_SPECS = camera_specs()
CAMERA_NAMES = list(CAMERA_SPECS.keys())
CAMERA_MATRICES = {name: camera_matrices(name) for name in CAMERA_NAMES}

print(f"Configured {len(CAMERA_NAMES)} synchronized cameras:")
for name in CAMERA_NAMES:
    print(f"  {name:>12}: eye={np.round(CAMERA_SPECS[name]['eye'], 3).tolist()}")


def render_rgb(client, view_name):
    view, proj = CAMERA_MATRICES[view_name]
    _, _, rgba, _, _ = p.getCameraImage(
        width=IMG_WIDTH,
        height=IMG_HEIGHT,
        viewMatrix=view,
        projectionMatrix=proj,
        renderer=p.ER_TINY_RENDERER,
        lightDirection=[-0.4, -0.6, -1.0],
        physicsClientId=client,
    )
    rgba = np.asarray(rgba, dtype=np.uint8).reshape(IMG_HEIGHT, IMG_WIDTH, 4)
    return rgba[:, :, :3]


Configured 12 synchronized cameras:
         front: eye=[0.0, -1.75, 1.15]
   front_right: eye=[1.237, -1.237, 1.15]
         right: eye=[1.75, -0.0, 1.15]
    back_right: eye=[1.237, 1.237, 1.15]
          back: eye=[0.0, 1.75, 1.15]
     back_left: eye=[-1.237, 1.237, 1.15]
          left: eye=[-1.75, 0.0, 1.15]
    front_left: eye=[-1.237, -1.237, 1.15]
           top: eye=[0.0, 0.0, 2.65]
   top_oblique: eye=[0.75, -0.75, 2.35]
  low_front_left: eye=[-0.9, -1.35, 0.87]
  low_back_right: eye=[0.9, 1.35, 0.87]


## Run Simulation and Record Twelve Videos

This cell advances the physics simulation at 480 Hz and records at 60 FPS. All 12 cameras are sampled at the same simulation ticks, so frame `k` from every video corresponds to the same physical time and state.


In [4]:
client = connect_pybullet()
table_id, ball_id, room_ids = create_tabletop_world(client)

video_paths = {name: OUTPUT_DIR / f"ping_pong_bounce_12view_{name}.mp4" for name in CAMERA_NAMES}
writers = {
    name: imageio.get_writer(
        path,
        fps=VIDEO_FPS,
        codec="libx264",
        quality=8,
        macro_block_size=16,
    )
    for name, path in video_paths.items()
}

trajectory = []

try:
    for step in range(N_STEPS + 1):
        t = step * TIME_STEP
        pos, orn = p.getBasePositionAndOrientation(ball_id, physicsClientId=client)
        lin_vel, ang_vel = p.getBaseVelocity(ball_id, physicsClientId=client)
        contacts = p.getContactPoints(bodyA=ball_id, bodyB=table_id, physicsClientId=client)
        trajectory.append([
            t,
            pos[0], pos[1], pos[2],
            lin_vel[0], lin_vel[1], lin_vel[2],
            ang_vel[0], ang_vel[1], ang_vel[2],
            len(contacts),
        ])

        if step % STEPS_PER_FRAME == 0:
            for name, writer in writers.items():
                writer.append_data(render_rgb(client, name))

        apply_ping_pong_air_drag(client, ball_id)
        p.stepSimulation(physicsClientId=client)
finally:
    for writer in writers.values():
        writer.close()
    p.disconnect(client)

trajectory = np.asarray(trajectory)
trajectory_path = OUTPUT_DIR / "ping_pong_bounce_12view_trajectory.csv"
np.savetxt(
    trajectory_path,
    trajectory,
    delimiter=",",
    header="time,x,y,z,vx,vy,vz,wx,wy,wz,num_table_contacts",
    comments="",
)

camera_path = OUTPUT_DIR / "ping_pong_bounce_12view_cameras.csv"
camera_rows = []
for name in CAMERA_NAMES:
    spec = CAMERA_SPECS[name]
    camera_rows.append([name, *spec["eye"], *spec["up"], spec["fov"]])
np.savetxt(
    camera_path,
    camera_rows,
    delimiter=",",
    fmt="%s",
    header="name,eye_x,eye_y,eye_z,up_x,up_y,up_z,fov_deg",
    comments="",
)

print("Saved videos:")
for name, path in video_paths.items():
    print(f"  {name:>12}: {path}")
print(f"Saved trajectory: {trajectory_path}")
print(f"Saved camera metadata: {camera_path}")


Saved videos:
         front: pybullet_outputs/ping_pong_bounce_12view_front.mp4
   front_right: pybullet_outputs/ping_pong_bounce_12view_front_right.mp4
         right: pybullet_outputs/ping_pong_bounce_12view_right.mp4
    back_right: pybullet_outputs/ping_pong_bounce_12view_back_right.mp4
          back: pybullet_outputs/ping_pong_bounce_12view_back.mp4
     back_left: pybullet_outputs/ping_pong_bounce_12view_back_left.mp4
          left: pybullet_outputs/ping_pong_bounce_12view_left.mp4
    front_left: pybullet_outputs/ping_pong_bounce_12view_front_left.mp4
           top: pybullet_outputs/ping_pong_bounce_12view_top.mp4
   top_oblique: pybullet_outputs/ping_pong_bounce_12view_top_oblique.mp4
  low_front_left: pybullet_outputs/ping_pong_bounce_12view_low_front_left.mp4
  low_back_right: pybullet_outputs/ping_pong_bounce_12view_low_back_right.mp4
Saved trajectory: pybullet_outputs/ping_pong_bounce_12view_trajectory.csv
Saved camera metadata: pybullet_outputs/ping_pong_bounce_12view_

## Quick Bounce Sanity Check

This reports impact times, rebound peaks, and approximate peak-height ratios. Because this is a ping-pong ball, the simulation includes air drag, so the vacuum contact time is only a reference. The checks give you simple evidence that the rendered sequence follows the physical assumptions rather than being only an animation.


In [5]:
times = trajectory[:, 0]
z = trajectory[:, 3]
vacuum_contact_time = np.sqrt(2 * ((BALL_START_Z - BALL_RADIUS) - TABLE_TOP_Z) / abs(GRAVITY))

def local_maxima(values):
    return np.flatnonzero((values[1:-1] > values[:-2]) & (values[1:-1] >= values[2:])) + 1

def local_minima(values):
    return np.flatnonzero((values[1:-1] < values[:-2]) & (values[1:-1] <= values[2:])) + 1

surface_center_z = TABLE_TOP_Z + BALL_RADIUS
impact_indices = local_minima(z)
impact_indices = impact_indices[z[impact_indices] <= surface_center_z + 0.01]
peak_indices = local_maxima(z)
rebound_peak_indices = peak_indices[times[peak_indices] > times[impact_indices[0]]] if len(impact_indices) else []
peak_heights = z[rebound_peak_indices] - (TABLE_TOP_Z + BALL_RADIUS)
peak_times = times[rebound_peak_indices]

if len(impact_indices):
    first_impact_time = times[impact_indices[0]]
    min_z = np.min(z)

    print(f"Vacuum first-contact estimate:    {vacuum_contact_time:.4f} s")
    print(f"First simulated impact:            {first_impact_time:.4f} s")
    print(f"Lowest ball center height:         {min_z:.4f} m")
    print(f"Tabletop + ball radius:            {TABLE_TOP_Z + BALL_RADIUS:.4f} m")
    print(f"Detected impacts:                  {len(impact_indices)}")

    print("\nImpact times:")
    for i, idx in enumerate(impact_indices[:8], start=1):
        print(f"  impact {i}: t={times[idx]:.3f} s, center_z={z[idx]:.4f} m")

    print("\nRebound peaks above the tabletop:")
    for i, (t_peak, h_peak) in enumerate(zip(peak_times[:6], peak_heights[:6]), start=1):
        print(f"  peak {i}: t={t_peak:.3f} s, height={h_peak:.3f} m")

    if len(peak_heights) >= 2:
        ratios = peak_heights[1:6] / peak_heights[:5]
        print("\nSuccessive peak-height ratios:", np.round(ratios, 3))
else:
    print("No table contact detected. Increase DURATION_SEC or lower BALL_START_Z.")

Vacuum first-contact estimate:    0.3988 s
First simulated impact:            0.4083 s
Lowest ball center height:         0.7682 m
Tabletop + ball radius:            0.7700 m
Detected impacts:                  7

Impact times:
  impact 1: t=0.408 s, center_z=0.7703 m
  impact 2: t=0.967 s, center_z=0.7682 m
  impact 3: t=1.450 s, center_z=0.7721 m
  impact 4: t=1.833 s, center_z=0.7702 m
  impact 5: t=2.123 s, center_z=0.7713 m
  impact 6: t=2.296 s, center_z=0.7704 m
  impact 7: t=2.375 s, center_z=0.7700 m

Rebound peaks above the tabletop:
  peak 1: t=0.683 s, height=0.381 m
  peak 2: t=1.206 s, height=0.284 m
  peak 3: t=1.640 s, height=0.181 m
  peak 4: t=1.977 s, height=0.102 m
  peak 5: t=2.208 s, height=0.037 m
  peak 6: t=2.333 s, height=0.007 m

Successive peak-height ratios: [0.745 0.638 0.565 0.366 0.197]


## Preview Videos

In [6]:
for name in CAMERA_NAMES:
    print(name)
    display(Video(str(video_paths[name]), embed=True, html_attributes="controls loop"))

front


front_right


right


back_right


back


...

front_left


top


top_oblique


low_front_left


low_back_right


## Notes for Tuning and Writeup Use

- Increase `SIM_HZ`, `numSolverIterations`, or `numSubSteps` for more stable contact resolution.
- Change `BALL_RESTITUTION`, `TABLE_RESTITUTION`, and friction constants to match a specific ball/table material pair.
- Change `DRAG_COEFFICIENT` or `AIR_DENSITY` if you want a different aerodynamic model.
- Increase `IMG_WIDTH`, `IMG_HEIGHT`, or `quality` for cleaner videos, at the cost of slower rendering.
- The generated trajectory CSV contains time, position, linear velocity, angular velocity, and contact counts for downstream analysis.
- The generated camera CSV records each virtual camera eye, up vector, and field of view, which helps describe the setup as calibrated multi-view observation.

For the project writeup, describe this as a controlled synthetic data generator: PyBullet produces a single physically constrained state trajectory, while the 12 virtual cameras produce synchronized observations from known viewpoints. This separates **physical validity** from **visual coverage**: the laws constrain the motion, and the camera rig increases how much of that motion can be observed. The two low-angle views are included because they emphasize contact and rebound geometry; unlike a bottom camera, they are not blocked by the opaque tabletop.
